In [0]:
import time, datetime

def subTime(
    t: float, 
    unit: str) -> float:
    """
    Subtracts a time interval from the current time.

    Parameters
    --------------------
    t : float
        The time interval to subtract.
    unit : str
        The unit of the time interval, such as h, min, sec, etc.
    
    Returns
    --------------------
    float
        The resulting time (seconds since epoch date).
    """

    # Acceptable units and their scales
    con = [
        {'unit': 'h', 'scale': 3600}, {'unit': 'hr', 'scale': 3600}, {'unit': 'hour', 'scale': 3600},
        {'unit': 'm', 'scale': 60}, {'unit': 'min', 'scale': 60}, {'unit': 'minute', 'scale': 60},
        {'unit': 's', 'scale': 1}, {'unit': 'sec', 'scale': 1}, {'unit': 'second', 'scale': 1}
        ]

    # Try scaling the given interval according to the unit specifier
    try:
        interval = t * [c['scale'] for c in con if c['unit'] == unit][0]

    # If the unit is not recognized, raise an exception with the acceptable units
    except IndexError:
        raise Exception("Invalid unit.  Please use one of the following: " + str([c['unit'] for c in con]) + ".")

    # Send the caller the current time minus the scaled interval
    return time.time() - interval


In [0]:
from pyspark.sql import DataFrame, functions as fn
from pyspark.sql.connect.session import SparkSession

def setProcessState(
    processName: str,
    stateName: str,
    spark: SparkSession
):
    """
    Adds a row to the control.process_execution_state table to record the current state of the process.

    Parameters
    --------------------
    processName : str
        The name of the process.
    stateName : str
        The name of the state.
    spark : SparkSession
        The caller's SparkSession, passed so this function can execute in the same context.
    """

    # Verify parameters were passed
    assert processName, "processName is required"
    assert stateName, "stateName is required"

    # Verify process and state exist
    r = spark.sql(f"select id from control.process where name = '{processName}'")
    assert r.count() > 0, f"No such process '{processName}'"
    pid = r.collect()[0][0]

    r = spark.sql(f"select id from control.execution_state where name = '{stateName}'")
    assert r.count() > 0, f"No such execution state '{stateName}'"
    sid = r.collect()[0][0]

    # Insert the process execution state
    spark.sql(f"insert into control.process_execution_state (process_id, state_id) values ({pid}, {sid})")
    r = spark.sql(f"select max(state_change_time) from control.process_execution_state where process_id = {pid} and state_id = {sid}")
    if r.count() > 0:
        ts = r.collect()[0][0]

    print(f"Process: {processName} ({pid}) changed to execution state {stateName} ({sid}) at {ts}")

# Shortcut functions for calling setProcessState with common execution states:
def startProcess(processName: str, spark: SparkSession):
    """
    Shortcut for setProcessState(processName, 'Started', spark)
    """
    setProcessState(processName, 'Started', spark)

def failProcess(processName: str, spark: SparkSession):
    """
    Shortcut for setProcessState(processName, 'Failed', spark)
    """
    setProcessState(processName, 'Failed', spark)

def succeedProcess(processName: str, spark: SparkSession):
    """
    Shortcut for setProcessState(processName, 'Succeeded', spark)
    """
    setProcessState(processName, 'Succeeded', spark)



In [0]:
setProcessState('Grout', 'Starting', spark)
time.sleep(1.8)

startProcess('Grout', spark)
time.sleep(0.2)

setProcessState('Grout', 'Running', spark)
time.sleep(3.2)

succeedProcess('Grout', spark)


In [0]:
# setProcessState(None, None, spark)
# setProcessState(None, 'None', spark)
# setProcessState('None', None, spark)
# setProcessState('None', 'None', spark)
# setProcessState('Tile', 'None', spark)


In [0]:
try:
    print(f"current time: {time.ctime()}")

    print(time.ctime(subTime(7.5, 'h')))

    print(time.ctime(subTime(7.5, 'whatever')))
except:
    print("Well, that didn't work...")

In [0]:
def processMayRun(
    processName: str, 
    validDate: float,
    spark: SparkSession
    ) -> {bool, str}:
    """
    Returns flag indicating if process dependencies are satisfied and a message describing the reason for the flag value.
    """

    if validDate < subTime(24, 'h'):
        return {
            'mayRun': False,
            'message': f"Process {processName} cannot run as dependent data is more than a day old."
        }
    else:
        return {
            'mayRun': True,
            'message': 'Sure, go ahead'}

In [0]:
processName = 'Grout'

intervals = [12, 24, 36]
for i in intervals:

    interval = subTime(i, 'h')
    o = processMayRun(processName, interval, spark)
    print(f"Interval: {i} hours\n\tProcess {processName} may run: {o['mayRun']}, reason: {o['message']}")
